In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import pickle
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE

In [2]:
def selectkbest(indep_x,dep_y,n):
    test=SelectKBest(score_func=chi2,k=n)
    fit1=test.fit(indep_x,dep_y)
    selectk_features=fit1.transform(indep_x)
    return selectk_features

In [3]:
def split_scaler(indep_x,dep_y):
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    x_train,x_test,y_train,y_test=train_test_split(indep_x,dep_y,test_size=0.25,random_state=42)
    sc=StandardScaler()
    x_train=sc.fit_transform(x_train)
    x_test=sc.transform(x_test)
    return x_train,x_test,y_train,y_test

In [4]:
def cm_prediction(classifier,x_test):
    y_pred=classifier.predict(x_test)
    from sklearn.metrics import confusion_matrix
    cm=confusion_matrix(y_test,y_pred)
    from sklearn.metrics import classification_report
    report=classification_report(y_test,y_pred)
    from sklearn.metrics import accuracy_score
    Accuracy=accuracy_score(y_test,y_pred)
    return classifier,Accuracy,report,cm,x_test,y_test

In [5]:
def logistic(x_train,y_train,x_test):
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import GridSearchCV
    param_grid={'solver':['lbfgs','newton-cg','liblinear']}
    classifier=GridSearchCV(LogisticRegression(),param_grid,refit=True,verbose=3,n_jobs=-1,scoring='f1_weighted')
    classifier.fit(x_train,y_train)
    classifier,Accuracy,report,cm,x_test,y_test=cm_prediction(classifier,x_test)
    return classifier,Accuracy,report,cm,x_test,y_test

In [6]:
def svc(x_train,y_train,x_test):
    from sklearn.svm import SVC
    from sklearn.model_selection import GridSearchCV
    param_grid={'kernel':['rbf','linear','poly'],'gamma':['auto','scale']}
    classifier=GridSearchCV(SVC(probability=True),param_grid,verbose=3,n_jobs=-1,scoring='f1_weighted')
    classifier.fit(x_train,y_train)
    classifier,Accuracy,report,cm,x_test,y_test=cm_prediction(classifier,x_test)
    return classifier,Accuracy,report,cm,x_test,y_test

In [7]:
def Decision(x_train,y_train,x_test):
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.model_selection import GridSearchCV
    param_grid={'criterion':['gini','entropy','log_loss'],'splitter':['best','random']}
    classifier=GridSearchCV(DecisionTreeClassifier(),param_grid,refit=True,verbose=3,n_jobs=-1,scoring='f1_weighted')
    classifier.fit(x_train,y_train)
    classifier,Accuracy,report,cm,x_test,y_test=cm_prediction(classifier,x_test)
    return classifier,Accuracy,report,cm,x_test,y_test

In [8]:
def random(x_train,y_train,x_test):
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import GridSearchCV
    param_grid={'criterion':['gini','entropy','log_loss'],'max_features':['sqrt','log2'],'n_estimators':[100]}
    classifier=GridSearchCV(RandomForestClassifier(),param_grid,refit=True,verbose=3,n_jobs=-1,scoring='f1_weighted')
    classifier.fit(x_train,y_train)
    classifier,Accuracy,report,cm,x_test,y_test=cm_prediction(classifier,x_test)
    return classifier,Accuracy,report,cm,x_test,y_test

In [9]:
def selectk_classification(acclog,accsvc,accdt,accrf):
    selectk_dataframe=pd.DataFrame(index=['chisquare'],columns=['Logistic','SVC','DecisionTree','RandomForest'])
    for number,idex in enumerate(selectk_dataframe.index):
        selectk_dataframe['Logistic'][idex]=acclog[number]
        selectk_dataframe['SVC'][idex]=accsvc[number]
        selectk_dataframe['DecisionTree'][idex]=accdt[number]
        selectk_dataframe['RandomForest'][idex]=accrf[number]
    return selectk_dataframe

In [10]:
dataset=pd.read_csv("preprocessed ILPD.csv")
df=dataset
df=pd.get_dummies(df,dtype=int,drop_first=True)
df

,Age,TB,DB,Alkphos,Sgpt,Sgot,TP,ALB,A/G Ratio,Selector,Gender_Male
0,65,0.7,0.10,187.00,16.0,18,6.8,3.3,0.90,1,0
1,62,5.3,2.95,481.75,64.0,100,7.5,3.2,0.74,1,1
2,62,5.3,2.95,481.75,60.0,68,7.0,3.3,0.89,1,1
3,58,1.0,0.40,182.00,14.0,20,6.8,3.4,1.00,1,1
4,72,3.9,2.00,195.00,27.0,59,7.3,2.4,0.40,1,1
...,...,...,...,...,...,...,...,...,...,...,...
578,60,0.5,0.10,481.75,20.0,34,5.9,1.6,0.37,0,1
579,40,0.6,0.10,98.00,35.0,31,6.0,3.2,1.10,1,1
580,52,0.8,0.20,245.00,48.0,49,6.4,3.2,1.00,1,1
581,31,1.3,0.50,184.00,29.0,32,6.8,3.4,1.00,1,1


In [11]:
indep_x=df.drop('Selector',axis=1)
dep_y=df['Selector']
indep_x.shape
dep_y.shape
print(indep_x.shape)
print(type(indep_x))


(583, 10)
<class 'pandas.core.frame.DataFrame'>


In [14]:
kbest=selectkbest(indep_x,dep_y,5)
acclog=[]
accsvc=[]
accdt=[]
accrf=[]
x_train, x_test, y_train, y_test=split_scaler(kbest,dep_y) 
x_train, x_test, y_train, y_test


(array([[ 1.07637252, -0.68881048,  0.29990792,  1.10970864,  2.11147647],
        [ 1.14090466,  2.03006912,  2.1229076 ,  2.04762906, -0.00471511],
        [ 0.56011538, -0.20545411, -0.64923835, -0.13076675, -0.1346567 ],
        ...,
        [-0.47239892, -0.74923003, -0.12903319,  1.44251911, -0.22747212],
        [-1.76304178, -0.87006912, -0.39369897, -0.61485471, -0.84005389],
        [-1.76304178, -0.62839094, -0.22942366, -0.37281073, -0.39453987]]),
 array([[-1.63397749, -0.3262932 , -0.64923835, -1.06868717, -0.74723847],
        [-2.08570249, -0.56797139,  2.1229076 ,  2.07032068,  0.77493442],
        [ 1.01184038,  2.03006912, -0.32068772,  2.07032068,  2.11147647],
        [-0.1497382 , -0.87006912, -0.79526086,  2.07032068,  0.77493442],
        [-0.27880249,  2.03006912,  0.99351481,  0.05076623,  0.16235265],
        [-0.73052749, -0.74923003,  0.06262135, -0.82664319, -0.67298613],
        [-1.31131678, -0.74923003, -0.55797429, -1.12919816, -1.04424781],
        [ 

In [15]:
classifier,Accuracy,report,cm,x_test,y_test=logistic(x_train,y_train,x_test)
acclog.append(Accuracy)
classifier,Accuracy,report,cm,x_test,y_test=svc(x_train,y_train,x_test)
accsvc.append(Accuracy)
classifier,Accuracy,report,cm,x_test,y_test=Decision(x_train,y_train,x_test)
accdt.append(Accuracy)
classifier,Accuracy,report,cm,x_test,y_test=random(x_train,y_train,x_test)
accrf.append(Accuracy)

result=selectk_classification(acclog,accsvc,accdt,accrf)

Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Fitting 5 folds for each of 6 candidates, totalling 30 fits


C:\Users\Sathish\AppData\Local\Temp\ipykernel_8640\1464381231.py:4: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  selectk_dataframe['Logistic'][idex]=acclog[number]
C:\Users\Sathish\AppData\Local\Temp\ipykernel_8640\1464381231.py:5: FutureWa

In [16]:
result
#5

,Logistic,SVC,DecisionTree,RandomForest
chisquare,0.760274,0.767123,0.684932,0.746575


In [21]:
result
#3

,Logistic,SVC,DecisionTree,RandomForest
chisquare,0.739726,0.746575,0.671233,0.739726


In [24]:
result
#7

,Logistic,SVC,DecisionTree,RandomForest
chisquare,0.753425,0.760274,0.69863,0.732877


In [27]:
result
#9

,Logistic,SVC,DecisionTree,RandomForest
chisquare,0.712329,0.753425,0.623288,0.732877
